# Combined Factors for Enhanced AI Alpha

This notebook demonstrates the full pipeline: data loading, alpha factor
construction, feature engineering, model training (Random Forest, Bagging,
NoOverlapVoter), and evaluation.

All core logic is imported from the `src/` package.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Data Pipeline

In [ ]:
from src.data.pipeline import register_bundle, load_bundle_and_engine, get_pricing
from config import UNIVERSE_END_DATE, FACTOR_LOOKBACK_YEARS, FACTOR_LOOKBACK_DAYS

register_bundle()
bundle_data, engine, trading_calendar, data_portal = load_bundle_and_engine()
print('Pipeline engine ready.')

## 2. Build Alpha Factors

In [ ]:
from src.factors import build_alpha_pipeline
from src.data.features import add_universal_features, add_regime_features
from zipline.pipeline.factors import Returns

pipeline, universe, sector = build_alpha_pipeline()

# Universal quant features
add_universal_features(pipeline, universe)

# Regime features
add_regime_features(pipeline, universe)

# Target: quantized 5-day forward returns
pipeline.add(Returns(window_length=5, mask=universe).quantiles(2), 'return_5d')
pipeline.add(Returns(window_length=5, mask=universe).quantiles(25), 'return_5d_p')

print('Pipeline built.')

## 3. Run Pipeline & Engineer Features

In [ ]:
from src.data.features import add_date_features, add_sector_dummies, add_target

universe_end = pd.Timestamp(UNIVERSE_END_DATE, tz='UTC')
factor_start = universe_end - pd.DateOffset(years=FACTOR_LOOKBACK_YEARS, days=FACTOR_LOOKBACK_DAYS)

all_factors = engine.run_pipeline(pipeline, factor_start, universe_end)

add_date_features(all_factors, factor_start, universe_end)
all_factors, sector_columns = add_sector_dummies(all_factors)
add_target(all_factors)

print(f'Factor DataFrame shape: {all_factors.shape}')
all_factors.head()

## 4. Train / Validation / Test Split

In [ ]:
from src.models import train_valid_test_split
from config import FEATURES, TRAIN_SIZE, VALID_SIZE, TEST_SIZE

features = FEATURES + sector_columns
temp = all_factors.dropna().copy()
X = temp[features]
y = temp['target']

X_train, X_valid, X_test, y_train, y_valid, y_test = train_valid_test_split(
    X, y, TRAIN_SIZE, VALID_SIZE, TEST_SIZE
)
print(f'Train: {len(X_train)}, Valid: {len(X_valid)}, Test: {len(X_test)}')

## 5. Random Forest Baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from src.utils import plot, rank_features_by_importance
from config import CLF_RANDOM_STATE, N_DAYS_PER_LEAF, N_STOCKS_PER_LEAF, TREE_SIZES

clf_parameters = {
    'criterion': 'entropy',
    'min_samples_leaf': N_STOCKS_PER_LEAF * N_DAYS_PER_LEAF,
    'oob_score': True,
    'n_jobs': -1,
    'random_state': CLF_RANDOM_STATE,
}

train_score, valid_score, oob_score_list, feat_imp = [], [], [], []

for n_trees in tqdm(TREE_SIZES, desc='Training RF'):
    clf = RandomForestClassifier(n_trees, **clf_parameters)
    clf.fit(X_train, y_train)
    train_score.append(clf.score(X_train, y_train.values))
    valid_score.append(clf.score(X_valid, y_valid.values))
    oob_score_list.append(clf.oob_score_)
    feat_imp.append(clf.feature_importances_)

plot([TREE_SIZES]*3, [train_score, valid_score, oob_score_list],
     ['train', 'validation', 'oob'],
     title='Accuracy vs Number of Trees', x_label='Trees', y_label='Accuracy')

print('\nFeatures Ranked by Average Importance:\n')
rank_features_by_importance(np.average(feat_imp, axis=0), features)

## 6. NoOverlapVoter Ensemble

In [ ]:
from src.models import NoOverlapVoter
from config import FINAL_N_TREES

clf = RandomForestClassifier(FINAL_N_TREES, **clf_parameters)
clf_nov = NoOverlapVoter(clf)
clf_nov.fit(pd.concat([X_train, X_valid]), pd.concat([y_train, y_valid]))

print(f'Train: {clf_nov.score(X_train, y_train.values):.4f}')
print(f'OOB:   {clf_nov.oob_score_:.4f}')
print(f'Valid: {clf_nov.score(X_valid, y_valid.values):.4f}')
print(f'Test:  {clf_nov.score(X_test, y_test.values):.4f}')

## 7. Evaluate AI Alpha vs Traditional Factors

In [ ]:
from src.models.evaluation import show_sample_results
from config import EVAL_FACTOR_NAMES

all_assets = all_factors.index.levels[1].values.tolist()
all_pricing = get_pricing(data_portal, trading_calendar, all_assets, factor_start, universe_end)

print('=== TEST SET ===')
show_sample_results(all_factors, X_test, clf_nov, EVAL_FACTOR_NAMES, all_pricing)